# Retail Customer Behavioral Analysis

This notebook documents the end-to-end ML workflow for churn prediction in an e-commerce retail dataset.

Coverage:
- Data exploration and quality checks
- Preprocessing outputs review
- PCA explained variance analysis
- Model comparison and best model summary

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid')
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / 'data' / 'raw' / 'retail_customers_COMPLETE_CATEGORICAL.csv'
DATA_TRAIN_TEST = PROJECT_ROOT / 'data' / 'train_test'
REPORTS = PROJECT_ROOT / 'reports'

In [ ]:
df = pd.read_csv(DATA_RAW)
print('Shape:', df.shape)
print('Columns:', len(df.columns))
df.head()

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_table = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_table = missing_table[missing_table['missing_count'] > 0]
missing_table.head(20)

In [ ]:
if 'Churn' in df.columns:
    churn_ratio = df['Churn'].value_counts(normalize=True).sort_index()
    print(churn_ratio)
    ax = df['Churn'].value_counts().plot(kind='bar', color=['#4C78A8', '#F58518'])
    ax.set_title('Churn Class Distribution')
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    plt.show()
else:
    print('Churn column not found in raw dataset.')

In [ ]:
X_train = pd.read_csv(DATA_TRAIN_TEST / 'X_train_processed.csv')
X_test = pd.read_csv(DATA_TRAIN_TEST / 'X_test_processed.csv')
y_train = pd.read_csv(DATA_TRAIN_TEST / 'y_train.csv')
y_test = pd.read_csv(DATA_TRAIN_TEST / 'y_test.csv')

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('y_train:', y_train.shape)
print('y_test :', y_test.shape)

In [ ]:
pca = PCA().fit(X_train)
cum_var = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(cum_var) + 1), cum_var, color='#2A9D8F')
plt.axhline(y=0.95, color='red', linestyle='--', label='95% variance')
plt.title('PCA Cumulative Explained Variance')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.legend()
plt.tight_layout()
plt.show()

n_components_95 = int(np.argmax(cum_var >= 0.95) + 1)
print('Components for >=95% variance:', n_components_95)

In [ ]:
comparison = pd.read_csv(REPORTS / 'model_comparison.csv')
display_cols = [
    'model_name', 'test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc', 'cv_best_score'
]
comparison[display_cols].sort_values('test_roc_auc', ascending=False)

In [ ]:
plot_df = comparison[['model_name', 'test_roc_auc', 'test_f1', 'test_accuracy']].melt(
    id_vars='model_name',
    var_name='metric',
    value_name='score'
)

plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x='model_name', y='score', hue='metric')
plt.title('Model Performance Comparison')
plt.ylim(0.0, 1.05)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
with open(REPORTS / 'training_summary.json', 'r', encoding='utf-8') as f:
    training_summary = json.load(f)

training_summary

## Deployment Quick Check

After training, run the API from the project root:

`python app/app.py`

Then call `/predict` with JSON customer features to get:
- churn_probability
- predicted_class